# P-Adic Embedding of CTX DEM Terrain

This notebook visualizes the CTX Digital Elevation Model (DEM) and its p-adic Sierpinski embedding using corrected parameters (s=0.5, m=0, p=3).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, Normalize
from pathlib import Path
import sys

# Setup path
sys.path.insert(0, '/Volumes/Fangorn/padic_fractal_analysis/src')
from padic.padic_embedding import embed_padic_cloud, get_paper_s

# Load DEM
cache_dir = Path('/Volumes/Fangorn/padic_fractal_analysis/cache')
dem_data = np.load(str(cache_dir / 'dem_clean.npy'))
dem_normalized = np.load(str(cache_dir / 'dem_normalized.npy'))

print(f"DEM loaded: {dem_data.shape}")
print(f"Elevation range: [{dem_data.min():.2f}, {dem_data.max():.2f}] m")

## DEM Visualization

In [ ]:
# Create hillshade visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(dem_data, vert_exag=0.1)

axes[0].imshow(hillshade, cmap='gray')
axes[0].set_title('DEM - Hillshaded', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Pixel X')
axes[0].set_ylabel('Pixel Y')

im = axes[1].imshow(dem_normalized, cmap='viridis')
axes[1].set_title('DEM - Elevation Normalized', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Pixel X')
axes[1].set_ylabel('Pixel Y')
plt.colorbar(im, ax=axes[1], label='Normalized Elevation')

plt.tight_layout()
plt.show()

## P-Adic Embeddings (Multi-Scale)

In [ ]:
# Load pre-computed embeddings
for scale_name in ['l=4', 'l=6']:
    embeddings = np.load(str(cache_dir / f'padic_embeddings_{scale_name}.npy'))
    grid_values = np.load(str(cache_dir / f'dem_grid_{scale_name}.npy'))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left: DEM grid with overlay
    cell_h = dem_data.shape[0] // int(np.sqrt(len(embeddings)))
    cell_w = dem_data.shape[1] // int(np.sqrt(len(embeddings)))
    
    ax1.imshow(hillshade, cmap='gray')
    grid_size = int(np.sqrt(len(embeddings)))
    for i in range(grid_size + 1):
        ax1.axhline(i * cell_h, color='cyan', linewidth=1, alpha=0.5)
        ax1.axvline(i * cell_w, color='cyan', linewidth=1, alpha=0.5)
    ax1.set_title(f'{scale_name}: DEM Grid', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Pixel X')
    ax1.set_ylabel('Pixel Y')
    
    # Right: P-Adic embedding
    scatter = ax2.scatter(embeddings[:, 0], embeddings[:, 1],
                         c=grid_values.flatten(), cmap='viridis', s=100,
                         alpha=0.8, edgecolors='black', linewidth=0.5)
    ax2.set_title(f'{scale_name}: P-Adic Sierpinski (s=0.5, m=0)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Real')
    ax2.set_ylabel('Imag')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax2, label='Mean Elevation (m)')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n{scale_name}:")
    print(f"  Regions: {len(embeddings)}")
    print(f"  Grid: {grid_size}×{grid_size}")
    print(f"  Embedding span: {np.sqrt((embeddings[:, 0].max()-embeddings[:, 0].min())**2 + (embeddings[:, 1].max()-embeddings[:, 1].min())**2):.3f}")